# 🧠 Trinity Cognitive Probes — Starter Notebook

This notebook shows you how to:
1. Load the dataset
2. Make baseline predictions
3. Submit to the leaderboard

**Expected runtime**: < 5 minutes (no API calls)

## 📦 Setup

Install dependencies and load libraries.

In [ ]:
# Install required packages
!pip install -q anthropic openai

import pandas as pd
import numpy as np
from pathlib import Path

## 📂 Load Data

In [ ]:
# Paths
DATA_DIR = Path("/kaggle/input/trinity-cognitive-probes")

# Load sample submission
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

# Load data files (if available)
try:
    metacognition_df = pd.read_csv(DATA_DIR / "tmp_metacognition.csv")
    print(f"✅ Loaded metacognition data: {len(metacognition_df)} items")
except FileNotFoundError:
    print("⚠️  Data files not in input directory")
    metacognition_df = None

# Preview submission format
print("\nSample submission format:")
print(sample_submission.head())
print(f"\nTotal items to predict: {len(sample_submission)}")

## 🎯 Tier 1: Random Baseline

This gives you a score on the leaderboard. Replace with actual predictions!

In [ ]:
# Create baseline submission
submission = sample_submission.copy()

# Strategy 1: All zeros (random guessing)
submission['score'] = 0.0

# Save
submission.to_csv("submission.csv", index=False)
print("✅ Baseline submission saved!")
print(f"   Score range: {submission['score'].min():.2f} to {submission['score'].max():.2f}")

## 🚀 Tier 2: Smart Baseline (Difficulty-Aware)

Use difficulty information to make better predictions.

In [ ]:
# This assumes you have the data files with difficulty info
if metacognition_df is not None:
    # Create a lookup dict for difficulty
    difficulty_map = dict(zip(metacognition_df['id'], metacognition_df['difficulty']))
    
    # Smart strategy: higher confidence for easier items
    def predict_score(item_id, difficulty):
        # Easier items = higher confidence = better score
        if difficulty < 5:
            return 0.5  # Easy items
        elif difficulty < 10:
            return 0.3  # Medium items
        else:
            return 0.1  # Hard items (be uncertain!)
    
    submission['score'] = submission['id'].apply(
        lambda x: predict_score(x, difficulty_map.get(x, 10))
    )
    
    submission.to_csv("submission.csv", index=False)
    print("✅ Smart baseline saved!")
    print(f"   Score range: {submission['score'].min():.2f} to {submission['score'].max():.2f}")
else:
    print("⚠️  Data files needed for smart baseline")

## 🤖 Tier 3: Using LLM APIs (Optional)

For actual predictions, you'll need API keys. Add them as Kaggle secrets.

In [ ]:
# Uncomment to use actual LLM APIs

# import os
# from kaggle.secrets import UserSecretsClient

# # Get API key from Kaggle secrets
# user_secrets = UserSecretsClient()
# API_KEY = user_secrets.get_secret("ANTHROPIC_API_KEY")

# # Make predictions
# # (See kaggle/eval/runner.py for full implementation)

print("🔒 API calls commented out — add your API keys to use")

## 📊 Understanding Your Score

Your score is based on:
- **Correctness**: Is the answer right?
- **Calibration**: Does confidence match accuracy?
- **Difficulty**: Hard items count more!

### Scoring Rules
```
+1: Correct + well-calibrated confidence
 0: Partial OR appropriate uncertainty
-1: Wrong OR overconfident (worst!)
```

### Key Tips
1. **Don't be overconfident!** Better to say "I'm not sure" than be wrong and confident.
2. **Use 5% confidence buckets**: 0.70, 0.75, 0.80 (not 0.73)
3. **Hard items matter**: They count 2x more than easy items.

See `docs/LEADERBOARD_STRATEGIES.md` for advanced tips!

## ✅ Submit

Your submission is ready! Click "Save Version" and then "Submit" to the competition.

In [ ]:
# Final check
print("Submission file:")
print(f"  Rows: {len(submission)}")
print(f"  Columns: {list(submission.columns)}")
print(f"  Score mean: {submission['score'].mean():.4f}")
print(f"  Score std: {submission['score'].std():.4f}")
print("\n✅ Ready to submit!")